In [1]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv(r"C:\Users\Admin\Downloads\teaching_prices_messy.csv")

In [14]:
df

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
...,...,...,...,...,...,...
92,2026-03-19,Pulse C,City 1,207.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
93,2026-03-19,Pulse C,City 2,213.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
94,2026-03-19,Sweetener D,City 1,178.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
95,2026-03-19,Sweetener D,City 2,184.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 4.7 KB


In [15]:
df.describe()

,date,item,city,price,unit,data_label
count,97,97,97,96,97,97
unique,12,5,2,70,1,1
top,2026-01-08,Staple A,City 1,202.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
freq,9,24,49,5,97,97


In [17]:
df['item'] = df['item'].str.strip()

In [18]:
df.loc[df['item'] == 'oil b', 'item'] = 'Staple A'

In [19]:
df = df.drop_duplicates()

In [20]:
df['price_num'] = pd.to_numeric(df['price'].astype(str).str.replace(',', '', regex=False), errors='coerce')

In [21]:
df.loc[df['price_num'] >= 1000, 'price_num'] = np.nan

In [22]:
df['date'] = pd.to_datetime(df['date'])

In [24]:
df = df.sort_values(['item', 'city', 'date'])

In [25]:
summary_stats = df.groupby('item')['price_num'].agg(
    mean='mean',
    median='median',
    count='count'
).reset_index()

print("--- Overall Item Statistics (Mean & Median) ---")
print(summary_stats)

--- Overall Item Statistics (Mean & Median) ---
          item        mean  median  count
0        Oil B  324.833333   324.0     24
1      Pulse C  205.913043   206.0     23
2     Staple A  111.260870   112.5     23
3  Sweetener D  167.152174   167.5     23


In [26]:
print("\n--- First-to-Last Percentage Change by Item ---")
for item, grp in df.groupby('item'):
    by_date = grp.groupby('date')['price_num'].mean().dropna()
    if len(by_date) >= 2:
        first_val = by_date.iloc[0]
        last_val = by_date.iloc[-1]
        pct_change = ((last_val - first_val) / first_val) * 100
        print(f"Item: {item}")
        print(f"  First Date ({by_date.index[0].date()}) Avg: {first_val:.2f}")
        print(f"  Last Date ({by_date.index[-1].date()}) Avg: {last_val:.2f}")
        print(f"  Percentage Change: {pct_change:.2f}%\n")


--- First-to-Last Percentage Change by Item ---
Item: Oil B
  First Date (2026-01-01) Avg: 301.00
  Last Date (2026-03-19) Avg: 347.00
  Percentage Change: 15.28%

Item: Pulse C
  First Date (2026-01-01) Avg: 200.00
  Last Date (2026-03-19) Avg: 210.50
  Percentage Change: 5.25%

Item: Staple A
  First Date (2026-01-01) Avg: 104.00
  Last Date (2026-03-19) Avg: 117.50
  Percentage Change: 12.98%

Item: Sweetener D
  First Date (2026-01-01) Avg: 152.00
  Last Date (2026-03-19) Avg: 181.50
  Percentage Change: 19.41%



In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\Admin\Downloads\teaching_prices_messy.csv')
df['item'] = df['item'].str.strip()
df.loc[df['item'] == 'oil b', 'item'] = 'Staple A'
df = df.drop_duplicates()
df['price_num'] = pd.to_numeric(df['price'].astype(str).str.replace(',', '', regex=False), errors='coerce')

df['is_flagged'] = df['price_num'].isna() | (df['price_num'] >= 1000)

print("--- WITHOUT flagged values (clean data only) ---")
clean_stats = df[~df['is_flagged']].groupby('item')['price_num'].agg(
    mean='mean',
    median='median',
    count='count'
).reset_index()
print(clean_stats)

print("\n--- WITH flagged values (including NaN and outliers) ---")
raw_stats = df.groupby('item')['price_num'].agg(
    mean='mean',
    median='median',
    count='count'
).reset_index()
print(raw_stats)

--- WITHOUT flagged values (clean data only) ---
          item        mean  median  count
0        Oil B  324.833333   324.0     24
1      Pulse C  205.913043   206.0     23
2     Staple A  111.260870   112.5     23
3  Sweetener D  167.152174   167.5     23

--- WITH flagged values (including NaN and outliers) ---
          item        mean  median  count
0        Oil B  324.833333  324.00     24
1      Pulse C  205.913043  206.00     23
2     Staple A  158.708333  112.50     24
3  Sweetener D  576.812500  167.75     24
